# Preparação dos Dados (Data Prep)

## Estudo de Caso: Qualidade na Central de Atendimento

Uma empresa de Contact Center fez uma pesquisa no momento da contratação de um grupo de colaboradores (agentes de atendimento) através do <a href="https://pt.surveymonkey.com/">Survey Monkey</a>. Após 6 meses da contratação verificou se quais colaboradores atingiram um bom desempenho de Qualidade e dos indicadores operacionais e estes agentes foram identificados com o valor 1 na coluna "Target".

Antes de criar um modelo que possa ser utilizado para entender os fatores mais associados a uma boa performance futura, o time de Atendimento ao Cliente precisa processar os dados brutos da ferramenta de pesquisa e realizar algumas análises exploratórias. Você como Cientista de Dados está encarregado desta tarefa de preparação dos dados brutos.

### Dados Brutos do Questionário

In [ ]:
import pandas as pd

In [ ]:
df_bruto = pd.read_csv('dados_contact_center_survey.csv', 
                       sep = ";", 
                       low_memory=False, 
                       header=[0,1])
df_bruto.head()

In [ ]:
df_bruto.shape

### Dados de Desempenho

In [ ]:
df_desempenho = pd.read_csv('dados_contact_center_survey_desempenho.csv', 
                            sep = ";", 
                            low_memory=False)
df_desempenho.head()

### Dados Esperados pelo time de Atendimento ao Cliente para a Análise Exploratória

In [ ]:
df_proc = pd.read_csv('dados_contact_center_survey_processado.csv', 
                      sep = ";")

In [ ]:
df_proc.head()

## Correção dos nomes das variáveis
Perceba que o dataframe orginal tem um problema no cabeçalho: temos duas linhas para definir o nome das variáveis com cada resposta dos colaboradores. Para corrigir isso, realizaremos algumas etapas utilizando diferentes funções do Pandas e vários recursos do Python. Por isso, é importante que o Cientista de Dados conheça diversas ferramentas diferentes para poder escolher a mais adequada para se utilizar em cada tarefa.

In [ ]:
df_bruto.head()

### Identificação do índice dos textos das questões.
Esse índice será utilizado nas etapas seguintes.

In [ ]:
indices = [i for i, c in enumerate(df_bruto.columns) if not c[0].startswith('Unnamed')]
indices

### Obtenção do texto da questão.
Esse texto será utilizado para atribuir os nomes às variáveis.

In [ ]:
questions = [c[0] for c in df_bruto.columns if not c[0].startswith('Unnamed')]
questions

### Definição das fatias do DataFrame.
Essas fatias serão utilizadas para selecionar os blocos de variáveis relacionados a cada questão.

In [ ]:
slices = [slice(i, j) for i, j in zip(indices, indices[1:] + [None])]
slices

### Pipeline de Preparação dos Dados
Em cada bloco de variáveis serão realizados os seguintes processamentos:
1. Atribuição do índice para unificação das perguntas
2. Empilhamento das diversas respostas possíveis: **pd.melt()**
3. Seleção dos registros com respostas

Ao final, teremos em cada linha os colaboradores (respondentes) e em cada coluna as perguntas.

In [ ]:
# Cria lista para armazenar cada fatia tratada do DataFrame
data = []

for i, q in enumerate(slices):

    # Obtém a variável Id_Agente para utilizar como índice
    d = pd.concat([df_bruto.iloc[:,0], df_bruto.iloc[:,slices[i]]], axis=1)
    
    # Empilha as diversas respostas possíveis em 1 variável
    d = d.melt(id_vars = 'Id_Agente', var_name=questions[i], col_level=1)
    
    # Seleciona apenas os registros que tiveram resposta
    d = d.loc[d['value'] == 1]
    
    # Define Id_Agente como índice
    d.index = d['Id_Agente']
    
    # Elimina as variáveis Id_Agente e value para evita repetições
    d.drop(['Id_Agente', 'value'], axis=1, inplace=True)
    
    # Adiciona a fatia na lista
    data.append(d)

In [ ]:
data[35]

### Preparação das variáveis extra questões

In [ ]:
df_prim = df_bruto.iloc[:,0:2].copy()
df_prim.columns = df_prim.columns.droplevel(0)
df_prim.index = df_prim['Id_Agente']
df_prim

### Unificação dos Dados

In [ ]:
data.insert(0, df_prim)

In [ ]:
data

In [ ]:
df_corrigido = pd.concat(data, axis=1)
df_corrigido.fillna('0 - Sem resposta', inplace=True)
df_corrigido.reset_index(drop=True, inplace=True)
df_corrigido

## Verificação de dados duplicados

In [ ]:
df_corrigido.duplicated().sum()

In [ ]:
df_corrigido = df_corrigido.drop_duplicates()
df_corrigido.shape

## Adição dos dados de desempenho
Utilizaremos o Pandas Merge para unificar a base de questões e a base de desempenho.

In [ ]:
df_completa = pd.merge(left=df_corrigido, 
                       right=df_desempenho, 
                       how='left', 
                       left_on='Id_Agente', right_on='Id_Agente')
df_completa

# Análise Exploratória dos Dados

## Utilizando o SweetViz

In [ ]:
# Instale o pacote em seu ambiente virtual com "pip install sweetviz"
import sweetviz as sv

# Executa o Data Profiling
aed_sv = sv.analyze(df_completa, 
                    target_feat='Target')

# Salva o arquivo HTML final com o relatório
aed_sv.show_html('aed_sv.html')

## Utilizando o Pandas Profiling

In [ ]:
# Instale o pacote em seu ambiente virtual com "pip install pandas_profiling".
import pandas_profiling

# Executa Data Profiling
aed_pp = df_completa.profile_report()

# Salva o arquivo HTML final com o relatório
aed_pp.to_file(output_file="aed_pp.html")

# Aplicação: Quais questões estão mais associadas ao bom desempenho dos agentes?
Identifique quais questões estão mais associadas ao bom desempenho dos agentes e explique como essa associação ocorre na base de dados.